In [2]:
import pandas as pd
import numpy as np

In [3]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv
Saving data.csv to data.csv


In [4]:
test_data = pd.read_csv('test.csv')
train_data = pd.read_csv('data.csv')

In [5]:
train_data['rating'].isnull().sum()

np.int64(0)

In [6]:
all_data = pd.concat([train_data, test_data])
n_users = all_data['user_id'].max()
n_items = all_data['item_id'].max()

print(f"\nتعداد کل کاربران: {n_users}")
print(f"تعداد کل فیلم‌ها: {n_items}")


تعداد کل کاربران: 943
تعداد کل فیلم‌ها: 1682


In [7]:
k = 10
alpha = 0.005
beta = 0.02
epochs = 50

print(f"تعداد ویژگی‌های پنهان (k): {k}")
print(f"نرخ یادگیری (alpha): {alpha}")
print(f"ضریب تنظیم (beta): {beta}")
print(f"تعداد تکرارها (epochs): {epochs}")

P = np.random.normal(loc=0, scale=0.1, size=(n_users + 1, k))
Q = np.random.normal(loc=0, scale=0.1, size=(n_items + 1, k))

print(f"\nشکل ماتریس کاربران (P): {P.shape}")
print(f"شکل ماتریس فیلم‌ها (Q): {Q.shape}")

تعداد ویژگی‌های پنهان (k): 10
نرخ یادگیری (alpha): 0.005
ضریب تنظیم (beta): 0.02
تعداد تکرارها (epochs): 50

شکل ماتریس کاربران (P): (944, 10)
شکل ماتریس فیلم‌ها (Q): (1683, 10)


In [8]:
training_history = []

print("شروع فرآیند آموزش مدل...")

for epoch in range(epochs):
  for user_id, item_id, rating, title in train_data.itertuples(index=False):
    user_vector = P[user_id]
    item_vector = Q[item_id]
    prediction = np.dot(user_vector, item_vector)

    error = rating - prediction

    P[user_id] += alpha * (error * item_vector - beta * user_vector)
    Q[item_id] += alpha * (error * user_vector - beta * item_vector)

  total_squared_error = 0

  for user_id, item_id, rating, title in train_data.itertuples(index=False):
    prediction = P[user_id] @ Q[item_id].T # ضرب داخلی
    total_squared_error += (rating - prediction)**2

  train_rmse = np.sqrt(total_squared_error / len(train_data))
  training_history.append(train_rmse)
  print(f"Epoch {epoch+1:02d}/{epochs}  |  Training RMSE: {train_rmse:.4f}")



print("\nفرآیند آموزش با موفقیت به پایان رسید!")




شروع فرآیند آموزش مدل...
Epoch 01/50  |  Training RMSE: 3.6875
Epoch 02/50  |  Training RMSE: 2.7298
Epoch 03/50  |  Training RMSE: 1.5792
Epoch 04/50  |  Training RMSE: 1.2393
Epoch 05/50  |  Training RMSE: 1.1013
Epoch 06/50  |  Training RMSE: 1.0336
Epoch 07/50  |  Training RMSE: 0.9955
Epoch 08/50  |  Training RMSE: 0.9715
Epoch 09/50  |  Training RMSE: 0.9548
Epoch 10/50  |  Training RMSE: 0.9421
Epoch 11/50  |  Training RMSE: 0.9318
Epoch 12/50  |  Training RMSE: 0.9229
Epoch 13/50  |  Training RMSE: 0.9148
Epoch 14/50  |  Training RMSE: 0.9071
Epoch 15/50  |  Training RMSE: 0.8998
Epoch 16/50  |  Training RMSE: 0.8927
Epoch 17/50  |  Training RMSE: 0.8857
Epoch 18/50  |  Training RMSE: 0.8789
Epoch 19/50  |  Training RMSE: 0.8723
Epoch 20/50  |  Training RMSE: 0.8658
Epoch 21/50  |  Training RMSE: 0.8595
Epoch 22/50  |  Training RMSE: 0.8534
Epoch 23/50  |  Training RMSE: 0.8474
Epoch 24/50  |  Training RMSE: 0.8416
Epoch 25/50  |  Training RMSE: 0.8360
Epoch 26/50  |  Training 

In [9]:
print("\nشروع ارزیابی مدل روی داده‌های آزمون...")
test_squared_error = 0


for user_id, item_id, rating, title in test_data.itertuples(index=False):
  user_vector = P[user_id]
  item_vector = Q[item_id]

  prediction = np.dot(user_vector, item_vector)

  prediction = np.clip(prediction, 1, 5)

  test_squared_error += (rating - prediction)**2

test_mse = test_squared_error / len(test_data)

test_rmse = np.sqrt(test_mse)

print(f"\nRMSE نهایی روی داده‌های آزمون: {test_rmse:.4f}")

if test_rmse < 0.94:
    print("تبریک! شما به هدف پروژه (RMSE < 0.94) رسیدید.")
else:
    print("مقدار RMSE هنوز بالاتر از هدف پروژه است. سعی کنید هایپرپارامترها (k, alpha, beta, epochs) را تغییر دهید.")


شروع ارزیابی مدل روی داده‌های آزمون...

RMSE نهایی روی داده‌های آزمون: 0.9367
تبریک! شما به هدف پروژه (RMSE < 0.94) رسیدید.


In [10]:
print("\nشروع تحلیل نتایج و محاسبه مقادیر تکین...")

R_hat = Q[1:] @ P[1:].T

print(f"ماتریس کامل پیش‌بینی R_hat با شکل {R_hat.shape} ساخته شد.")

U, s, Vt = np.linalg.svd(R_hat, full_matrices=False)

print("تجزیه SVD با موفقیت روی R_hat انجام شد.")
print(f"تعداد مقادیر تکین به دست آمده: {len(s)}")

print("\n۱۰ مقدار تکین اول (به ترتیب نزولی):")
print(s[:10])


شروع تحلیل نتایج و محاسبه مقادیر تکین...
ماتریس کامل پیش‌بینی R_hat با شکل (1682, 943) ساخته شد.
تجزیه SVD با موفقیت روی R_hat انجام شد.
تعداد مقادیر تکین به دست آمده: 943

۱۰ مقدار تکین اول (به ترتیب نزولی):
[3822.58828061  224.38387762  143.70048757  126.35864689  113.53439509
  100.07362078   96.03580539   89.48294084   85.02905482   84.18154097]


In [11]:
print("\nمحاسبه انرژی و سهم هر مؤلفه...")

energy = s**2
total_energy = np.sum(energy)
explained_variance_ratio = energy / total_energy
cumulative_variance = np.cumsum(explained_variance_ratio)

print("\nتحلیل برای ۱۰ مؤلفه اول:")
print("="*30)
for i in range(10):
  print(f"مؤلفه {i+1:02d}:")
  print(f"  - سهم از کل واریانس: {explained_variance_ratio[i]:.2%}")
  print(f"  - واریانس تجمعی: {cumulative_variance[i]:.2%}")



محاسبه انرژی و سهم هر مؤلفه...

تحلیل برای ۱۰ مؤلفه اول:
مؤلفه 01:
  - سهم از کل واریانس: 99.04%
  - واریانس تجمعی: 99.04%
مؤلفه 02:
  - سهم از کل واریانس: 0.34%
  - واریانس تجمعی: 99.38%
مؤلفه 03:
  - سهم از کل واریانس: 0.14%
  - واریانس تجمعی: 99.52%
مؤلفه 04:
  - سهم از کل واریانس: 0.11%
  - واریانس تجمعی: 99.63%
مؤلفه 05:
  - سهم از کل واریانس: 0.09%
  - واریانس تجمعی: 99.72%
مؤلفه 06:
  - سهم از کل واریانس: 0.07%
  - واریانس تجمعی: 99.79%
مؤلفه 07:
  - سهم از کل واریانس: 0.06%
  - واریانس تجمعی: 99.85%
مؤلفه 08:
  - سهم از کل واریانس: 0.05%
  - واریانس تجمعی: 99.90%
مؤلفه 09:
  - سهم از کل واریانس: 0.05%
  - واریانس تجمعی: 99.95%
مؤلفه 10:
  - سهم از کل واریانس: 0.05%
  - واریانس تجمعی: 100.00%
